# pymgcv Quick Start

This notebook demonstrates the core pymgcv API — fitting a GAM, inspecting the summary,
plotting smooth terms, and making predictions.

**Prerequisites:** `pip install pymgcv matplotlib`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymgcv import GAM, plot_smooth

np.random.seed(42)

## 1. Simulate Data

Create a dataset with two nonlinear effects and Gaussian noise.

In [ ]:
n = 500
x1 = np.random.uniform(0, 1, n)
x2 = np.random.uniform(0, 1, n)
y = np.sin(2 * np.pi * x1) + 0.5 * x2**2 + 0.3 * np.random.randn(n)

df = pd.DataFrame({"x1": x1, "x2": x2, "y": y})
df.head()

## 2. Fit a GAM

Use an R-style formula with `s()` for smooth terms. REML is the default (and recommended)
smoothing parameter selection method.

In [ ]:
model = GAM("y ~ s(x1) + s(x2)", data=df, method="REML")
model.fit()
print(model.summary())

## 3. Smooth Term Plots

Visualize the estimated smooth effects with 95% confidence bands.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_smooth(model, var_name="x1", ax=axes[0], ci=0.95)
plot_smooth(model, var_name="x2", ax=axes[1], ci=0.95)
plt.tight_layout()
plt.show()

## 4. Predictions

Generate predictions on the response scale.

In [ ]:
df["predicted"] = model.predict(df, scale="response")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["y"], df["predicted"], alpha=0.4, s=10)
ax.plot([df["y"].min(), df["y"].max()], [df["y"].min(), df["y"].max()], "r--")
ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.set_title("Actual vs Predicted")
plt.tight_layout()
plt.show()

## 5. Model Diagnostics

Check basis dimension adequacy with `gam_check()`.

In [ ]:
model.gam_check()

## 6. R Equivalent

The same model in R mgcv:

```r
library(mgcv)
model <- gam(y ~ s(x1) + s(x2), data = df, method = "REML")
summary(model)
plot(model, pages = 1)
```

pymgcv produces numerically equivalent output (EDF, coefficients, predictions match within 1%).